SECTION 0 — INSTALL DEPENDENCIES

In [ ]:
import subprocess, sys
 
def _run_install(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[INSTALL WARNING] {result.stderr[-300:]}")
    else:
        print(f"[INSTALL OK] {cmd[:70]}")
 
import torch
_TORCH_VER = torch.__version__.split('+')[0]
_CUDA_TAG  = ('cpu' if not torch.cuda.is_available()
               else 'cu' + torch.version.cuda.replace('.','')[:3])
print(f"[ENV] Torch {_TORCH_VER}  |  CUDA tag: {_CUDA_TAG}")
 
_PYG_URL = f"https://data.pyg.org/whl/torch-{_TORCH_VER}+{_CUDA_TAG}.html"
_run_install(
    f"pip install torch_scatter torch_sparse torch_cluster "
    f"torch_spline_conv torch_geometric -f {_PYG_URL} -q"
)
_run_install("pip install xgboost joblib -q")
print("[INSTALL] All dependencies ready.\n")

[ENV] Torch 2.10.0  |  CUDA tag: cu128
[INSTALL OK] pip install torch_scatter torch_sparse torch_cluster torch_spline_conv
[INSTALL OK] pip install xgboost joblib -q
[INSTALL] All dependencies ready.



SECTION 1 — IMPORTS, LOGGING & CONFIG

In [ ]:
import os, sys, random, warnings, time, pickle, logging
import gzip, shutil, urllib.request
from pathlib import Path
from functools import partial
 
warnings.filterwarnings('ignore')
 
for _h in logging.root.handlers[:]:
    logging.root.removeHandler(_h)
logging.basicConfig(
    level    = logging.INFO,
    format   = '%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt  = '%H:%M:%S',
    handlers = [logging.StreamHandler(sys.stdout)]
)
log = logging.getLogger('HGNN')
 
import numpy  as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors  as mcolors
import seaborn as sns
from tqdm import tqdm
 
from scipy.stats import spearmanr, kendalltau, wilcoxon
from sklearn.model_selection   import train_test_split, KFold
from sklearn.preprocessing     import StandardScaler, MinMaxScaler
from sklearn.metrics           import mean_squared_error, r2_score
from sklearn.linear_model      import Ridge
 
from joblib import Parallel, delayed
 
import torch
import torch.nn            as nn
import torch.nn.functional as F
import torch.optim         as optim
from torch_geometric.utils import from_networkx
from torch_geometric.nn    import SAGEConv, GATConv, BatchNorm
 
from xgboost import XGBRegressor
 
CFG = {
    'EPI_BETAS'         : [0.1, 0.2, 0.3],   
    'EPI_GAMMA'         : 0.10,              
    'EPI_SIGMA'         : 0.15,               
    'EPI_RUNS_FB'       : 15,                
    'EPI_RUNS_OTHER'    : 8,                  
    'EPI_STEPS_FB'      : 12,                 
    'EPI_STEPS_OTHER'   : 8,                  
    'EPI_N_JOBS'        : -1,   
    'GNN_EPOCHS'        : 300,
    'GNN_LR'            : 5e-3,
    'GNN_WD'            : 1e-4,
    'GNN_PATIENCE'      : 30,
    'GNN_HIDDEN'        : 64,
    'GNN_OUT'           : 32,
    'GNN_HEADS'         : 4,
    'GNN_DROPOUT'       : 0.3,
    'GNN_RANK_LAMBDA'   : 0.05,              
    'GNN_RANK_PAIRS'    : 256,  
    'XGB_TREES'         : 600,
    'XGB_DEPTH'         : 6,
    'XGB_LR'            : 0.05,
    'XGB_EARLY'         : 30,
    'XGB_SUBSAMPLE'     : 0.8,
    'XGB_COLSAMPLE'     : 0.8,
    'XGB_REG_ALPHA'     : 0.1,
    'XGB_REG_LAMBDA'    : 1.0,
    'XGB_MIN_CHILD_W'   : 3,
    'XGB_GAMMA'         : 0.1,
    'K_VALS'            : [50, 100, 200],
    'CV_FOLDS'          : 5,
    'ALPHA'             : 0.05,              
    'SEED'              : 42,
    'OUT_DIR'           : '/kaggle/working/results',
    'CACHE_DIR'         : '/kaggle/working/cache',
    'TRAIN_FRAC'        : 0.60,
    'VAL_FRAC'          : 0.20,
    'TEST_FRAC'         : 0.20,
}
 
SEED = CFG['SEED']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
 
OUT_DIR   = Path(CFG['OUT_DIR'])
CACHE_DIR = Path(CFG['CACHE_DIR'])
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 
PALETTE = {
    'blue'   : '#2E86AB',
    'red'    : '#E84855',
    'green'  : '#3BB273',
    'purple' : '#7B2D8B',
    'orange' : '#F18F01',
    'teal'   : '#44BBA4',
    'navy'   : '#1D3557',
    'gold'   : '#F4A261',
}
COLORS = list(PALETTE.values())
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.25,
    'grid.linestyle'   : '--',
    'figure.dpi'       : 120,
})
try:
    plt.rcParams['savefig.dpi'] = 150
except KeyError:
    pass
 
log.info(f'Device    = {DEVICE}')
log.info(f'Output    = {OUT_DIR}')
log.info(f'Cache     = {CACHE_DIR}')
log.info('Section 1 complete ✓')
print('Section 1 complete')

03:40:00  INFO      Device    = cuda
03:40:00  INFO      Output    = /kaggle/working/results
03:40:00  INFO      Cache     = /kaggle/working/cache
03:40:00  INFO      Section 1 complete ✓
Section 1 complete


SECTION 2 — DATASET DOWNLOAD & GRAPH LOADING

In [ ]:
log.info('=' * 60)
log.info('SECTION 2: Dataset Download & Graph Loading')
log.info('=' * 60)
 
DATASETS = {
    'facebook' : {
        'url'      : 'https://snap.stanford.edu/data/facebook_combined.txt.gz',
        'gz'       : str(CACHE_DIR / 'facebook_combined.txt.gz'),
        'txt'      : str(CACHE_DIR / 'facebook_combined.txt'),
        'directed' : False,
        'proxy_n'  : 4039,
        'proxy_m'  : 5,
    },
    'email' : {
        'url'      : 'https://snap.stanford.edu/data/email-Eu-core.txt.gz',
        'gz'       : str(CACHE_DIR / 'email-Eu-core.txt.gz'),
        'txt'      : str(CACHE_DIR / 'email-Eu-core.txt'),
        'directed' : True,
        'proxy_n'  : 1005,
        'proxy_m'  : 5,
    },
    'wikivote' : {
        'url'      : 'https://snap.stanford.edu/data/wiki-Vote.txt.gz',
        'gz'       : str(CACHE_DIR / 'wiki-Vote.txt.gz'),
        'txt'      : str(CACHE_DIR / 'wiki-Vote.txt'),
        'directed' : True,
        'proxy_n'  : 7115,
        'proxy_m'  : 5,
    },
}
 
 
def _download_dataset(name: str, info: dict) -> None:
    if Path(info['txt']).exists():
        log.info(f'[{name}] Already cached at {info["txt"]}')
        return
    log.info(f'[{name}] Downloading from SNAP...')
    try:
        urllib.request.urlretrieve(info['url'], info['gz'])
        with gzip.open(info['gz'], 'rb') as fin, \
             open(info['txt'], 'wb') as fout:
            shutil.copyfileobj(fin, fout)
        log.info(f'[{name}] Saved → {info["txt"]}')
    except Exception as exc:
        log.warning(f'[{name}] Download failed: {exc}')
 
 
def load_graph(name: str, info: dict) -> nx.Graph:
    _download_dataset(name, info)
 
    G_raw = None
    if Path(info['txt']).exists():
        try:
            create_using = nx.DiGraph() if info['directed'] else nx.Graph()
            G_raw = nx.read_edgelist(
                info['txt'], nodetype=int,
                create_using=create_using, comments='#')
            if info['directed']:
                G_raw = G_raw.to_undirected()
        except Exception as exc:
            log.warning(f'[{name}] Read failed: {exc}')
            G_raw = None
    if G_raw is None:
        log.warning(f'[{name}] Using BA proxy (n={info["proxy_n"]}, m={info["proxy_m"]})')
        G_raw = nx.barabasi_albert_graph(info['proxy_n'], info['proxy_m'], seed=SEED)
 
    G_raw = nx.Graph(G_raw)                        
    G_raw.remove_edges_from(nx.selfloop_edges(G_raw))
    lcc   = max(nx.connected_components(G_raw), key=len)
    G_out = nx.convert_node_labels_to_integers(
                G_raw.subgraph(lcc).copy(), ordering='sorted')
 
    assert not G_out.is_directed(),           f'{name}: graph is still directed'
    assert nx.is_connected(G_out),            f'{name}: graph is not connected'
    assert nx.number_of_selfloops(G_out) == 0, f'{name}: self-loops present'
 
    n, e = G_out.number_of_nodes(), G_out.number_of_edges()
    density = nx.density(G_out)
    avg_deg = sum(d for _, d in G_out.degree()) / n
    log.info(
        f'[{name}] n={n:,}  e={e:,}  '
        f'density={density:.5f}  avg_deg={avg_deg:.2f}'
    )
    return G_out
 
t_load = time.time()
G_fb = load_graph('facebook', DATASETS['facebook'])
G_em = load_graph('email',    DATASETS['email'])
G_wv = load_graph('wikivote', DATASETS['wikivote'])
log.info(f'All graphs loaded in {time.time()-t_load:.1f}s')
 
G        = G_fb
nodes    = sorted(G.nodes())
N        = len(nodes)
E        = G.number_of_edges()
node_idx = {n: i for i, n in enumerate(nodes)}
 
dataset_summary = pd.DataFrame([
    {'Dataset': 'Facebook', 'Nodes': G_fb.number_of_nodes(),
     'Edges': G_fb.number_of_edges(),
     'Avg Degree': f'{2*G_fb.number_of_edges()/G_fb.number_of_nodes():.2f}',
     'Directed': False},
    {'Dataset': 'Email-EU', 'Nodes': G_em.number_of_nodes(),
     'Edges': G_em.number_of_edges(),
     'Avg Degree': f'{2*G_em.number_of_edges()/G_em.number_of_nodes():.2f}',
     'Directed': True},
    {'Dataset': 'WikiVote', 'Nodes': G_wv.number_of_nodes(),
     'Edges': G_wv.number_of_edges(),
     'Avg Degree': f'{2*G_wv.number_of_edges()/G_wv.number_of_nodes():.2f}',
     'Directed': True},
])
dataset_summary.to_csv(OUT_DIR / 'dataset_summary.csv', index=False)
print(dataset_summary.to_string(index=False))
log.info('Section 2 complete ✓')
print('Section 2 complete')

03:43:21  INFO      ============================================================
03:43:21  INFO      SECTION 2: Dataset Download & Graph Loading
03:43:21  INFO      ============================================================
03:43:22  INFO      [facebook] Already cached at /kaggle/working/cache/facebook_combined.txt
03:43:22  INFO      [facebook] n=4,039  e=88,234  density=0.01082  avg_deg=43.69
03:43:22  INFO      [email] Downloading from SNAP...
03:43:23  INFO      [email] Saved → /kaggle/working/cache/email-Eu-core.txt
03:43:23  INFO      [email] n=986  e=16,064  density=0.03308  avg_deg=32.58
03:43:23  INFO      [wikivote] Downloading from SNAP...
03:43:24  INFO      [wikivote] Saved → /kaggle/working/cache/wiki-Vote.txt
03:43:25  INFO      [wikivote] n=7,066  e=100,736  density=0.00404  avg_deg=28.51
03:43:25  INFO      All graphs loaded in 3.4s
 Dataset  Nodes  Edges Avg Degree  Directed
Facebook   4039  88234      43.69     False
Email-EU    986  16064      32.58      True
Wiki

SECTION 3 — CENTRALITY FEATURES (8 features)

In [ ]:
log.info('=' * 60)
log.info('SECTION 3: Computing 8 Centrality Features')
log.info('=' * 60)
 
FEAT_NAMES = [
    'degree',       
    'betweenness',  
    'closeness',    
    'katz',         
    'kshell',       
    'clustering',   
    'voterank',     
    'harmonic',     
]
 
 
def compute_centralities(
        G_in: nx.Graph,
        name: str,
        seed: int = 42
) -> tuple[np.ndarray, list[str], dict]:

    cache_npy = CACHE_DIR / f'cent_{name}.npy'
    cache_pkl = CACHE_DIR / f'cent_{name}.pkl'
 
    if cache_npy.exists() and cache_pkl.exists():
        log.info(f'[{name}] Loading cached centralities')
        feats_raw = np.load(str(cache_npy))
        with open(cache_pkl, 'rb') as fh:
            cent_dict = pickle.load(fh)
        return feats_raw, FEAT_NAMES, cent_dict
 
    t0        = time.time()
    nodes_in  = sorted(G_in.nodes())
    N_g       = len(nodes_in)
    K_BET     = min(100, N_g)
 
    log.info(f'[{name}] [1/8] Degree centrality...')
    deg_c = nx.degree_centrality(G_in)
 
    log.info(f'[{name}] [2/8] Betweenness (k={K_BET})...')
    bet_c = nx.betweenness_centrality(
        G_in, k=K_BET, normalized=True, seed=seed)
 
    log.info(f'[{name}] [3/8] Closeness centrality...')
    clo_c = nx.closeness_centrality(G_in)
 
    log.info(f'[{name}] [4/8] Katz centrality (numpy)...')
    try:
        katz_raw = nx.katz_centrality_numpy(G_in, alpha=0.005)
        mx_k = max(abs(v) for v in katz_raw.values()) or 1.0
        katz_c = {n: v / mx_k for n, v in katz_raw.items()}
    except Exception as ex:
        log.warning(f'[{name}] Katz failed ({ex}) → degree proxy')
        katz_c = dict(deg_c)
 
    log.info(f'[{name}] [5/8] K-shell (core number)...')
    ks_c   = nx.core_number(G_in)
    max_ks = max(ks_c.values()) or 1
 
    log.info(f'[{name}] [6/8] Clustering coefficient...')
    cl_c = nx.clustering(G_in)
 
    log.info(f'[{name}] [7/8] VoteRank...')
    vote_spreaders = nx.voterank(G_in)
    vote_c  = {nd: 0.0 for nd in nodes_in}
    n_vote  = max(len(vote_spreaders), 1)
    for rank, nd in enumerate(vote_spreaders):
        vote_c[nd] = 1.0 - rank / n_vote
 
    log.info(f'[{name}] [8/8] Harmonic centrality...')
    harm_c   = nx.harmonic_centrality(G_in)
    max_harm = max(harm_c.values()) or 1.0
 
    feats_raw = np.array([
        [deg_c[nd],
         bet_c[nd],
         clo_c[nd],
         katz_c[nd],
         ks_c[nd]   / max_ks,
         cl_c[nd],
         vote_c[nd],
         harm_c[nd] / max_harm]
        for nd in nodes_in
    ], dtype=np.float32)
 
    feats_raw = np.nan_to_num(feats_raw, nan=0.0, posinf=1.0, neginf=0.0)
    feats_raw = np.clip(feats_raw, 0.0, None)
 
    cent_dict = {
        'deg': deg_c,  'bet': bet_c, 'clo': clo_c,  'katz': katz_c,
        'ks' : ks_c,   'cl' : cl_c,  'vote': vote_c, 'harm': harm_c,
        'max_ks': max_ks, 'max_harm': max_harm,
        'nodes': nodes_in,
    }
 
    np.save(str(cache_npy), feats_raw)
    with open(cache_pkl, 'wb') as fh:
        pickle.dump(cent_dict, fh)
    log.info(
        f'[{name}] Centralities done in {time.time()-t0:.1f}s  '
        f'→ cached'
    )
    return feats_raw, FEAT_NAMES, cent_dict
 
 
features_raw, feature_names, cent_fb = compute_centralities(
    G, name='facebook', seed=SEED)
 
deg_c  = cent_fb['deg'];   bet_c  = cent_fb['bet']
clo_c  = cent_fb['clo'];   katz_c = cent_fb['katz']
ks_c   = cent_fb['ks'];    cl_c   = cent_fb['cl']
vote_c = cent_fb['vote'];  harm_c = cent_fb['harm']
max_ks   = cent_fb['max_ks']
max_harm = cent_fb['max_harm']
 
cent_scaler = MinMaxScaler()
features    = cent_scaler.fit_transform(features_raw).astype(np.float32)
features    = np.clip(features, 0.0, 1.0)
 
log.info(
    f'Feature matrix: {features.shape}  '
    f'range=[{features.min():.4f}, {features.max():.4f}]'
)
print(f'  Feature names : {feature_names}')
print(f'  Shape         : {features.shape}')
 
feat_stats = pd.DataFrame(features, columns=feature_names).describe().T
feat_stats.to_csv(OUT_DIR / 'centrality_feature_stats.csv')
log.info('Section 3 complete ✓')
print('Section 3 complete')

03:43:29  INFO      ============================================================
03:43:29  INFO      SECTION 3: Computing 8 Centrality Features
03:43:29  INFO      ============================================================
03:43:29  INFO      [facebook] [1/8] Degree centrality...
03:43:29  INFO      [facebook] [2/8] Betweenness (k=100)...
03:43:31  INFO      [facebook] [3/8] Closeness centrality...
03:43:54  INFO      [facebook] [4/8] Katz centrality (numpy)...
03:43:55  INFO      [facebook] [5/8] K-shell (core number)...
03:43:55  INFO      [facebook] [6/8] Clustering coefficient...
03:43:57  INFO      [facebook] [7/8] VoteRank...
03:45:35  INFO      [facebook] [8/8] Harmonic centrality...
03:46:00  INFO      [facebook] Centralities done in 151.4s  → cached
03:46:00  INFO      Feature matrix: (4039, 8)  range=[0.0000, 1.0000]
  Feature names : ['degree', 'betweenness', 'closeness', 'katz', 'kshell', 'clustering', 'voterank', 'harmonic']
  Shape         : (4039, 8)
03:46:00  INFO    

SECTION 4 — EPIDEMIC SIMULATION LABELS

In [ ]:
log.info('=' * 60)
log.info('SECTION 4: Epidemic Simulation Labels (Optimized)')
log.info('=' * 60)
 
def _sir_single(G_in, start, beta, gamma, steps, N_g):
    """One MC run of SIR. Returns |R|/N."""
    S = set(G_in.nodes()) - {start}
    I = {start}
    R = set()
    for _ in range(steps):
        new_I, new_R = set(), set()
        for nd in I:
            if random.random() < gamma:
                new_R.add(nd)
            else:
                for nb in G_in.neighbors(nd):
                    if nb in S and random.random() < beta:
                        new_I.add(nb)
        S -= new_I
        I  = (I - new_R) | new_I
        R |= new_R
        if not I:
            break
    return len(R) / N_g
 
 
def _sis_single(G_in, start, beta, gamma, steps, N_g):
    S    = set(G_in.nodes()) - {start}
    I    = {start}
    peak = 1
    for _ in range(steps):
        new_I, new_S = set(), set()
        for nd in I:
            if random.random() < gamma:
                new_S.add(nd)
            else:
                for nb in G_in.neighbors(nd):
                    if nb in S and random.random() < beta:
                        new_I.add(nb)
        S  = (S - new_I) | new_S
        I  = (I - new_S) | new_I
        if len(I) > peak:
            peak = len(I)
        if not I:
            break
    return peak / N_g
 
 
def _seir_single(G_in, start, beta, sigma, gamma, steps, N_g):
    S = set(G_in.nodes()) - {start}
    E = set()
    I = {start}
    R = set()
    for _ in range(steps):
        new_E, new_I, new_R = set(), set(), set()
        for nd in I:
            if random.random() < gamma:
                new_R.add(nd)
            else:
                for nb in G_in.neighbors(nd):
                    if nb in S and random.random() < beta:
                        new_E.add(nb)
        for nd in list(E):
            if random.random() < sigma:
                new_I.add(nd)
        S -= new_E
        E  = (E - new_I) | new_E
        I  = (I - new_R) | new_I
        R |= new_R
        if not I and not E:
            break
    return len(R) / N_g
 
 
def _node_epi_score(
        nd, G_in, betas, gamma, sigma, steps, runs, N_g
) -> float:

    scores = []
    for b in betas:
        sir_s  = sum(_sir_single(G_in, nd, b, gamma, steps, N_g)
                     for _ in range(runs)) / runs
        sis_s  = sum(_sis_single(G_in, nd, b, gamma, steps, N_g)
                     for _ in range(runs)) / runs
        seir_s = sum(_seir_single(G_in, nd, b, sigma, gamma, steps, N_g)
                     for _ in range(runs)) / runs
        scores.extend([sir_s, sis_s, seir_s])
    return float(np.mean(scores))
 
 
def compute_epidemic_labels(
        G_in: nx.Graph,
        nodes_list: list,
        betas: list,
        gamma: float,
        sigma: float,
        steps: int,
        runs:  int,
        cache_name: str,
        n_jobs: int = -1,
) -> np.ndarray:

    cache_path = CACHE_DIR / f'labels_epi_{cache_name}.npy'
    if cache_path.exists():
        log.info(f'[{cache_name}] Loading cached epidemic labels')
        return np.load(str(cache_path))
 
    t0  = time.time()
    N_g = len(nodes_list)
    log.info(
        f'[{cache_name}] Simulating {N_g:,} nodes  '
        f'models=SIR+SIS+SEIR  β={betas}  '
        f'steps={steps}  runs={runs}  jobs={n_jobs}'
    )
 
    worker = partial(
        _node_epi_score,
        G_in=G_in, betas=betas, gamma=gamma, sigma=sigma,
        steps=steps, runs=runs, N_g=N_g,
    )
 
    raw_scores = Parallel(n_jobs=n_jobs, backend='loky', verbose=5)(
        delayed(worker)(nd) for nd in nodes_list
    )
 
    labels_out = np.array(raw_scores, dtype=np.float32)
 
    mn, mx = labels_out.min(), labels_out.max()
    if mx > mn:
        labels_out = (labels_out - mn) / (mx - mn)
    else:
        labels_out = np.zeros_like(labels_out)
 
    labels_out = np.clip(labels_out, 0.0, 1.0)
    np.save(str(cache_path), labels_out)
 
    elapsed = time.time() - t0
    log.info(
        f'[{cache_name}] Labels done in {elapsed:.1f}s  '
        f'mean={labels_out.mean():.4f}  '
        f'std={labels_out.std():.4f}  '
        f'range=[{labels_out.min():.4f}, {labels_out.max():.4f}]'
    )
    return labels_out
 
 
t_epi = time.time()
labels = compute_epidemic_labels(
    G_in       = G,
    nodes_list = nodes,
    betas      = CFG['EPI_BETAS'],
    gamma      = CFG['EPI_GAMMA'],
    sigma      = CFG['EPI_SIGMA'],
    steps      = CFG['EPI_STEPS_FB'],
    runs       = CFG['EPI_RUNS_FB'],
    cache_name = 'facebook',
    n_jobs     = CFG['EPI_N_JOBS'],
)
t_epi_total = time.time() - t_epi
 
log.info(
    f'Facebook labels: mean={labels.mean():.4f}  '
    f'std={labels.std():.4f}  '
    f'computed in {t_epi_total:.1f}s'
)
print(f'  Label range: [{labels.min():.5f}, {labels.max():.5f}]')
print(f'  Label std  : {labels.std():.5f}')
log.info('Section 4 complete ✓')
print('Section 4 complete')

03:46:30  INFO      ============================================================
03:46:30  INFO      SECTION 4: Epidemic Simulation Labels (Optimized)
03:46:30  INFO      ============================================================
03:46:30  INFO      [facebook] Simulating 4,039 nodes  models=SIR+SIS+SEIR  β=[0.1, 0.2, 0.3]  steps=12  runs=15  jobs=-1


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   15.5s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:  3.0min
[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:  5.3min
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed:  8.9min
[Parallel(n_jobs=-1)]: Done 640 tasks      | elapsed: 13.8min
[Parallel(n_jobs=-1)]: Done 874 tasks      | elapsed: 16.3min
[Parallel(n_jobs=-1)]: Done 1144 tasks      | elapsed: 23.6min
[Parallel(n_jobs=-1)]: Done 1450 tasks      | elapsed: 32.4min
[Parallel(n_jobs=-1)]: Done 1792 tasks      | elapsed: 42.3min
[Parallel(n_jobs=-1)]: Done 2170 tasks      | elapsed: 53.5min
[Parallel(n_jobs=-1)]: Done 2584 tasks      | elapsed: 66.1min
[Parallel(n_jobs=-1)]: Done 3034 tasks      | elapsed: 77.2min
[Parallel(n_jobs=-1)]: Done 3520 tasks      | elapsed: 88.0min


05:22:18  INFO      [facebook] Labels done in 5748.3s  mean=0.7006  std=0.1660  range=[0.0000, 1.0000]
05:22:18  INFO      Facebook labels: mean=0.7006  std=0.1660  computed in 5748.3s
  Label range: [0.00000, 1.00000]
  Label std  : 0.16604
05:22:19  INFO      Section 4 complete ✓
Section 4 complete


[Parallel(n_jobs=-1)]: Done 4039 out of 4039 | elapsed: 95.8min finished


SECTION 5 — PyTorch Geometric Data + Inductive Split

In [ ]:
log.info('=' * 60)
log.info('SECTION 5: PyG Data Object + Inductive Split (60/20/20)')
log.info('=' * 60)
 
data = from_networkx(G)
data.x = torch.tensor(features, dtype=torch.float).to(DEVICE)
data.y = torch.tensor(labels,   dtype=torch.float).to(DEVICE)
 
all_idx  = np.arange(N)
train_idx, temp_idx = train_test_split(
    all_idx, test_size=(1 - CFG['TRAIN_FRAC']), random_state=SEED)
val_idx, test_idx   = train_test_split(
    temp_idx,
    test_size=CFG['TEST_FRAC'] / (CFG['VAL_FRAC'] + CFG['TEST_FRAC']),
    random_state=SEED)
 
train_mask = torch.zeros(N, dtype=torch.bool)
val_mask   = torch.zeros(N, dtype=torch.bool)
test_mask  = torch.zeros(N, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True
 
data.train_mask = train_mask.to(DEVICE)
data.val_mask   = val_mask.to(DEVICE)
data.test_mask  = test_mask.to(DEVICE)
data            = data.to(DEVICE)
 
log.info(
    f'Split: Train={int(train_mask.sum())}  '
    f'Val={int(val_mask.sum())}  Test={int(test_mask.sum())}'
)
assert len(set(train_idx) & set(val_idx))  == 0, 'LEAK: train ∩ val'
assert len(set(train_idx) & set(test_idx)) == 0, 'LEAK: train ∩ test'
assert len(set(val_idx)   & set(test_idx)) == 0, 'LEAK: val ∩ test'
log.info('No data leakage detected ✓')
log.info('Section 5 complete ✓')
print('Section 5 complete')

05:24:26  INFO      ============================================================
05:24:26  INFO      SECTION 5: PyG Data Object + Inductive Split (60/20/20)
05:24:26  INFO      ============================================================
05:24:29  INFO      Split: Train=2423  Val=808  Test=808
05:24:29  INFO      No data leakage detected ✓
05:24:29  INFO      Section 5 complete ✓
Section 5 complete


SECTION 6 — HYBRID GNN ARCHITECTURE

In [ ]:
log.info('=' * 60)
log.info('SECTION 6: Hybrid GNN Architecture')
log.info('=' * 60)
 
 
class HybridGNN(nn.Module):
    def __init__(
        self,
        in_dim:  int,
        hidden:  int = 64,
        out_dim: int = 32,
        heads:   int = 4,
        dropout: float = 0.3,
    ):
        super().__init__()
        if hidden % heads != 0:
            raise ValueError(f'hidden ({hidden}) must be divisible by heads ({heads})')
 
        self.dropout = dropout
 
        self.sage1 = SAGEConv(in_dim, hidden)
        self.bn1   = BatchNorm(hidden)
 
        self.gat = GATConv(
            hidden, hidden // heads,
            heads=heads, dropout=dropout, add_self_loops=True)
        self.bn2 = BatchNorm(hidden)
 
        self.sage2 = SAGEConv(hidden, out_dim)
        self.bn3   = BatchNorm(out_dim)
 
        self.proj = nn.Linear(in_dim, out_dim, bias=False)
 
        self.head = nn.Sequential(
            nn.Linear(out_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(16, 1),
            nn.Sigmoid(),         
        )
 
        self._init_weights()
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
 
    def forward(self, data_in):
        x, ei = data_in.x, data_in.edge_index
 
        res = self.proj(x)
 
        x = F.relu(self.bn1(self.sage1(x, ei)))
        x = F.dropout(x, p=self.dropout, training=self.training)
 
        x = F.relu(self.bn2(self.gat(x, ei)))
        x = F.dropout(x, p=self.dropout, training=self.training)
 
        x = F.relu(self.bn3(self.sage2(x, ei)))
 
        embeddings = x + res                  
 
        scores = self.head(embeddings).squeeze(-1)  
 
        return embeddings, scores
 
 
model_gnn = HybridGNN(
    in_dim  = features.shape[1],
    hidden  = CFG['GNN_HIDDEN'],
    out_dim = CFG['GNN_OUT'],
    heads   = CFG['GNN_HEADS'],
    dropout = CFG['GNN_DROPOUT'],
).to(DEVICE)
 
n_params = sum(p.numel() for p in model_gnn.parameters())
n_trainable = sum(p.numel() for p in model_gnn.parameters() if p.requires_grad)
log.info(f'GNN parameters: total={n_params:,}  trainable={n_trainable:,}')
print(f'  Parameters: {n_params:,}')
print(model_gnn)
log.info('Section 6 complete ✓')
print('Section 6 complete')

05:24:32  INFO      ============================================================
05:24:32  INFO      SECTION 6: Hybrid GNN Architecture
05:24:32  INFO      ============================================================
05:24:33  INFO      GNN parameters: total=11,681  trainable=11,681
  Parameters: 11,681
HybridGNN(
  (sage1): SAGEConv(8, 64, aggr=mean)
  (bn1): BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (gat): GATConv(64, 16, heads=4)
  (bn2): BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (sage2): SAGEConv(64, 32, aggr=mean)
  (bn3): BatchNorm(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (proj): Linear(in_features=8, out_features=32, bias=False)
  (head): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): GELU(approximate='none')
  

SECTION 7 — TRAIN GNN

In [ ]:
log.info('=' * 60)
log.info('SECTION 7: Training GNN')
log.info('=' * 60)
 
 
def ranking_loss(
        scores: torch.Tensor,
        y:      torch.Tensor,
        idx:    torch.Tensor,
        n_pairs: int,
        device:  torch.device,
) -> torch.Tensor:
    ia = torch.randint(0, len(idx), (n_pairs,), device=device)
    ib = torch.randint(0, len(idx), (n_pairs,), device=device)
    diff_true = y[idx[ia]] - y[idx[ib]]
    diff_pred = scores[idx[ia]] - scores[idx[ib]]
    loss = -torch.log(
        torch.sigmoid(diff_pred * torch.sign(diff_true) + 1e-8)
    ).mean()
    return loss
 
 
optimizer_gnn = optim.Adam(
    model_gnn.parameters(),
    lr=CFG['GNN_LR'],
    weight_decay=CFG['GNN_WD'],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_gnn,
    T_max=CFG['GNN_EPOCHS'],
    eta_min=1e-5,
)
 
PATIENCE       = CFG['GNN_PATIENCE']
best_val_loss  = float('inf')
best_state     = None
patience_count = 0
train_losses   = []
val_losses     = []
 
tr_idx_t = data.train_mask.nonzero(as_tuple=True)[0]
 
t_gnn_train = time.time()
 
for epoch in range(1, CFG['GNN_EPOCHS'] + 1):
 
    model_gnn.train()
    optimizer_gnn.zero_grad()
    _, scores = model_gnn(data)
 
    mse_loss  = F.mse_loss(scores[data.train_mask], data.y[data.train_mask])
    rank_loss = ranking_loss(
        scores, data.y, tr_idx_t,
        n_pairs=CFG['GNN_RANK_PAIRS'], device=DEVICE)
    loss = mse_loss + CFG['GNN_RANK_LAMBDA'] * rank_loss
 
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_gnn.parameters(), 1.0)
    optimizer_gnn.step()
    scheduler.step()
 
    model_gnn.eval()
    with torch.no_grad():
        _, val_scores = model_gnn(data)
        val_loss = F.mse_loss(
            val_scores[data.val_mask],
            data.y[data.val_mask]
        ).item()
 
    train_losses.append(mse_loss.item())
    val_losses.append(val_loss)
 
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_state     = {k: v.detach().clone()
                          for k, v in model_gnn.state_dict().items()}
        patience_count = 0
    else:
        patience_count += 1
 
    if patience_count >= PATIENCE:
        log.info(f'Early stopping at epoch {epoch}  best_val={best_val_loss:.6f}')
        break
 
    if epoch % 50 == 0:
        log.info(
            f'Epoch {epoch:3d}  '
            f'train={mse_loss.item():.6f}  '
            f'val={val_loss:.6f}  '
            f'rank={rank_loss.item():.6f}  '
            f'patience={patience_count}/{PATIENCE}'
        )
 
t_gnn_total = time.time() - t_gnn_train
 
model_gnn.load_state_dict(best_state)
model_gnn.eval()
with torch.no_grad():
    emb_t, sc_t  = model_gnn(data)
    embeddings   = emb_t.detach().cpu().numpy()
    gnn_scores   = sc_t.detach().cpu().numpy()
 
log.info(
    f'Training complete  best_val={best_val_loss:.6f}  '
    f'time={t_gnn_total:.1f}s'
)
print(f'\n  Training complete | best val={best_val_loss:.6f}')
print(f'  Embedding shape  : {embeddings.shape}')
print(f'  GNN score range  : [{gnn_scores.min():.4f}, {gnn_scores.max():.4f}]')
log.info('Section 7 complete ✓')
print('Section 7 complete')


05:25:00  INFO      ============================================================
05:25:00  INFO      SECTION 7: Training GNN
05:25:00  INFO      ============================================================
05:25:01  INFO      Epoch  50  train=0.004210  val=0.002950  rank=0.624274  patience=1/30
05:25:02  INFO      Epoch 100  train=0.003727  val=0.002979  rank=0.612080  patience=22/30
05:25:02  INFO      Early stopping at epoch 108  best_val=0.002791
05:25:02  INFO      Training complete  best_val=0.002791  time=1.6s

  Training complete | best val=0.002791
  Embedding shape  : (4039, 32)
  GNN score range  : [0.0802, 0.9142]
05:25:02  INFO      Section 7 complete ✓
Section 7 complete


SECTION 8 — HYBRID FEATURE MATRIX (no data leakage)

In [ ]:
log.info('=' * 60)
log.info('SECTION 8: Hybrid Feature Matrix')
log.info('=' * 60)
 
X_fb   = np.concatenate([features, embeddings], axis=1).astype(np.float32)
y_fb   = labels
 
y_train = y_fb[train_idx]
y_val   = y_fb[val_idx]
y_test  = y_fb[test_idx]
 
feat_scaler = StandardScaler()
X_train     = feat_scaler.fit_transform(X_fb[train_idx])
X_val       = feat_scaler.transform(X_fb[val_idx])
X_test_raw  = feat_scaler.transform(X_fb[test_idx])
X_fb_all    = feat_scaler.transform(X_fb)
 
log.info(
    f'Hybrid feature matrix: {X_fb.shape}  '
    f'(cent={features.shape[1]}, emb={embeddings.shape[1]})'
)
print(f'  Shape: {X_fb.shape}  (train={len(X_train)}, val={len(X_val)}, test={len(X_test_raw)})')
log.info('Section 8 complete ✓')
print('Section 8 complete')

05:25:05  INFO      ============================================================
05:25:05  INFO      SECTION 8: Hybrid Feature Matrix
05:25:05  INFO      ============================================================
05:25:05  INFO      Hybrid feature matrix: (4039, 40)  (cent=8, emb=32)
  Shape: (4039, 40)  (train=2423, val=808, test=808)
05:25:05  INFO      Section 8 complete ✓
Section 8 complete


SECTION 9 — XGBOOST REGRESSOR

In [ ]:
log.info('=' * 60)
log.info('SECTION 9: XGBoost Regressor')
log.info('=' * 60)
 
t_xgb = time.time()
xgb_model = XGBRegressor(
    n_estimators          = CFG['XGB_TREES'],
    max_depth             = CFG['XGB_DEPTH'],
    learning_rate         = CFG['XGB_LR'],
    subsample             = CFG['XGB_SUBSAMPLE'],
    colsample_bytree      = CFG['XGB_COLSAMPLE'],
    reg_alpha             = CFG['XGB_REG_ALPHA'],
    reg_lambda            = CFG['XGB_REG_LAMBDA'],
    min_child_weight      = CFG['XGB_MIN_CHILD_W'],
    gamma                 = CFG['XGB_GAMMA'],
    random_state          = SEED,
    n_jobs                = -1,
    verbosity             = 0,
    early_stopping_rounds = CFG['XGB_EARLY'],
    eval_metric           = 'rmse',
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
t_xgb_total = time.time() - t_xgb
 
preds_all  = np.clip(xgb_model.predict(X_fb_all), 0.0, 1.0)
preds_test = preds_all[test_idx]
 
log.info(
    f'XGBoost done in {t_xgb_total:.1f}s  '
    f'best_iter={xgb_model.best_iteration}  '
    f'test_range=[{preds_test.min():.4f}, {preds_test.max():.4f}]'
)
print(f'  Best iteration  : {xgb_model.best_iteration}')
print(f'  Test pred range : [{preds_test.min():.4f}, {preds_test.max():.4f}]')
log.info('Section 9 complete ✓')
print('Section 9 complete')

05:25:13  INFO      ============================================================
05:25:13  INFO      SECTION 9: XGBoost Regressor
05:25:13  INFO      ============================================================
05:25:14  INFO      XGBoost done in 0.3s  best_iter=160  test_range=[0.1701, 0.8642]
  Best iteration  : 160
  Test pred range : [0.1701, 0.8642]
05:25:14  INFO      Section 9 complete ✓
Section 9 complete


SECTION 10 — EVALUATION METRICS

In [ ]:
log.info('=' * 60)
log.info('SECTION 10: Evaluation Metrics')
log.info('=' * 60)
 
 
def top_k_jaccard(y_true, y_pred, k, global_idx):
    k_ = min(k, len(y_true))
    top_t = set(global_idx[np.argsort(y_true)[-k_:]])
    top_p = set(global_idx[np.argsort(y_pred)[-k_:]])
    union = top_t | top_p
    return len(top_t & top_p) / len(union) if union else 0.0
 
 
def ndcg_at_k(y_true, y_pred, k):
    k_ = min(k, len(y_true))
    pred_ord = np.argsort(y_pred)[::-1][:k_]
    true_ord = np.argsort(y_true)[::-1][:k_]
    dcg  = sum(float(y_true[pred_ord[i]]) / np.log2(i + 2)
               for i in range(k_))
    idcg = sum(float(y_true[true_ord[i]]) / np.log2(i + 2)
               for i in range(k_))
    return dcg / idcg if idcg > 0 else 0.0
 
 
def top_k_accuracy(y_true, y_pred, k, global_idx):
    k_ = min(k, len(y_true))
    return len(
        set(global_idx[np.argsort(y_true)[-k_:]]) &
        set(global_idx[np.argsort(y_pred)[-k_:]])
    ) / k_
 
 
def monotonicity_index(scores):
    return len(np.unique(np.round(scores, 8))) / len(scores)
 
 
def precision_at_k(y_true, y_pred, k, threshold=0.5):
    k_ = min(k, len(y_true))
    top_pred = np.argsort(y_pred)[-k_:]
    return np.mean(y_true[top_pred] >= threshold)
 
 
def average_precision(y_true, y_pred, threshold=0.5):
    order = np.argsort(y_pred)[::-1]
    y_bin = (y_true >= threshold).astype(int)
    hits  = 0
    sum_p = 0.0
    for i, idx in enumerate(order):
        if y_bin[idx]:
            hits  += 1
            sum_p += hits / (i + 1)
    return sum_p / max(hits, 1)
 
 
def evaluate_full(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        global_idx: np.ndarray,
        k_vals: list,
        tag: str,
) -> dict:

    y_true = np.clip(y_true.astype(np.float64), 0.0, 1.0)
    y_pred = np.clip(y_pred.astype(np.float64), 0.0, 1.0)
 
    sp,  sp_p  = spearmanr(y_true, y_pred)
    kt,  kt_p  = kendalltau(y_true, y_pred)
    r2_v       = r2_score(y_true, y_pred)
    mse_v      = mean_squared_error(y_true, y_pred)
    rmse_v     = float(np.sqrt(mse_v))
    mi_v       = monotonicity_index(y_pred)
    map_v      = average_precision(y_true, y_pred)
 
    res = {
        'tag': tag,
        'spearman': sp, 'spearman_p': sp_p,
        'kendall':  kt, 'kendall_p':  kt_p,
        'r2': r2_v, 'mse': mse_v, 'rmse': rmse_v,
        'monotonicity': mi_v, 'map': map_v,
    }
 
    print(f'\n[{tag}]')
    print(f'  Spearman ρ  : {sp:.4f}  (p={sp_p:.2e})')
    print(f'  Kendall τ   : {kt:.4f}  (p={kt_p:.2e})')
    print(f'  R²          : {r2_v:.4f}')
    print(f'  MSE / RMSE  : {mse_v:.8f} / {rmse_v:.6f}')
    print(f'  Monotonicity: {mi_v:.4f}')
    print(f'  MAP         : {map_v:.4f}')
 
    for k in k_vals:
        j    = top_k_jaccard(y_true, y_pred, k, global_idx)
        nd   = ndcg_at_k(y_true, y_pred, k)
        ac   = top_k_accuracy(y_true, y_pred, k, global_idx)
        pk   = precision_at_k(y_true, y_pred, k)
        res[f'jaccard@{k}']   = j
        res[f'ndcg@{k}']      = nd
        res[f'topkacc@{k}']   = ac
        res[f'precision@{k}'] = pk
        print(
            f'  Jaccard@{k:<4}: {j:.4f}  '
            f'NDCG@{k}: {nd:.4f}  '
            f'TopK@{k}: {ac:.4f}  '
            f'P@{k}: {pk:.4f}'
        )
    return res
 
y_all        = data.y.detach().cpu().numpy()
test_mask_np = data.test_mask.detach().cpu().numpy()
idx_test     = np.where(test_mask_np)[0]
y_test_eval  = y_all[idx_test]
preds_test_eval = preds_all[idx_test]
 
main_metrics = evaluate_full(
    y_test_eval, preds_test_eval, idx_test, CFG['K_VALS'],
    'Facebook-test (Hybrid)'
)
 
spear = main_metrics['spearman']
ktau  = main_metrics['kendall']
r2    = main_metrics['r2']
mse   = main_metrics['mse']
mi    = main_metrics['monotonicity']
 
pd.DataFrame([main_metrics]).to_csv(
    OUT_DIR / 'main_test_metrics.csv', index=False)
 
log.info('Section 10 complete ✓')
print('Section 10 complete')

05:25:32  INFO      ============================================================
05:25:32  INFO      SECTION 10: Evaluation Metrics
05:25:32  INFO      ============================================================

[Facebook-test (Hybrid)]
  Spearman ρ  : 0.9169  (p=0.00e+00)
  Kendall τ   : 0.7583  (p=5.55e-228)
  R²          : 0.9228
  MSE / RMSE  : 0.00217181 / 0.046603
  Monotonicity: 0.8181
  MAP         : 0.9995
  Jaccard@50  : 0.4706  NDCG@50: 0.9621  TopK@50: 0.6400  P@50: 1.0000
  Jaccard@100 : 0.5038  NDCG@100: 0.9744  TopK@100: 0.6700  P@100: 1.0000
  Jaccard@200 : 0.6393  NDCG@200: 0.9805  TopK@200: 0.7800  P@200: 1.0000
05:25:32  INFO      Section 10 complete ✓
Section 10 complete


SECTION 11 — 5-FOLD CROSS-VALIDATION

In [ ]:
log.info('=' * 60)
log.info('SECTION 11: 5-Fold Cross-Validation')
log.info('=' * 60)
 
cv = KFold(n_splits=CFG['CV_FOLDS'], shuffle=True, random_state=SEED)
cv_results = {m: [] for m in ['spearman', 'kendall', 'r2', 'mse', 'ndcg50', 'jaccard50']}
 
for fold, (tr_i, va_i) in enumerate(cv.split(X_fb_all), 1):
    xgb_cv = XGBRegressor(
        n_estimators=350, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, verbosity=0,
    )
    xgb_cv.fit(X_fb_all[tr_i], y_fb[tr_i])
    p_va   = np.clip(xgb_cv.predict(X_fb_all[va_i]), 0.0, 1.0)
    g_idx  = np.arange(len(va_i))
    sp_, _ = spearmanr(y_fb[va_i], p_va)
    kt_, _ = kendalltau(y_fb[va_i], p_va)
    r2_    = r2_score(y_fb[va_i], p_va)
    mse_   = mean_squared_error(y_fb[va_i], p_va)
    nd50   = ndcg_at_k(y_fb[va_i], p_va, 50)
    jac50  = top_k_jaccard(y_fb[va_i], p_va, 50, g_idx)
    for key, val in zip(
        ['spearman', 'kendall', 'r2', 'mse', 'ndcg50', 'jaccard50'],
        [sp_, kt_, r2_, mse_, nd50, jac50]
    ):
        cv_results[key].append(val)
    log.info(
        f'Fold {fold}:  ρ={sp_:.4f}  τ={kt_:.4f}  '
        f'R²={r2_:.4f}  NDCG@50={nd50:.4f}'
    )
 
print('\n  5-Fold CV Results:')
for k, v in cv_results.items():
    print(f'  {k:<12}: {np.mean(v):.4f} ± {np.std(v):.4f}')
 
cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(OUT_DIR / 'cv_results.csv', index=False)
log.info('Section 11 complete ✓')
print('Section 11 complete')

05:25:37  INFO      ============================================================
05:25:37  INFO      SECTION 11: 5-Fold Cross-Validation
05:25:37  INFO      ============================================================
05:25:39  INFO      Fold 1:  ρ=0.9293  τ=0.7775  R²=0.9293  NDCG@50=0.9693
05:25:41  INFO      Fold 2:  ρ=0.9271  τ=0.7720  R²=0.9292  NDCG@50=0.9594
05:25:42  INFO      Fold 3:  ρ=0.9259  τ=0.7738  R²=0.9388  NDCG@50=0.9592
05:25:44  INFO      Fold 4:  ρ=0.9355  τ=0.7891  R²=0.9359  NDCG@50=0.9651
05:25:46  INFO      Fold 5:  ρ=0.9407  τ=0.7970  R²=0.9408  NDCG@50=0.9681

  5-Fold CV Results:
  spearman    : 0.9317 ± 0.0056
  kendall     : 0.7819 ± 0.0096
  r2          : 0.9348 ± 0.0048
  mse         : 0.0018 ± 0.0001
  ndcg50      : 0.9642 ± 0.0042
  jaccard50   : 0.3534 ± 0.0534
05:25:46  INFO      Section 11 complete ✓
Section 11 complete


SECTION 12 — BASELINE COMPARISONS

In [ ]:
log.info('=' * 60)
log.info('SECTION 12: Baseline Comparisons')
log.info('=' * 60)
 
gnn_sc  = StandardScaler()
X_emb_tr = gnn_sc.fit_transform(embeddings[train_idx])
X_emb_te = gnn_sc.transform(embeddings[test_idx])
ridge   = Ridge(alpha=1.0)
ridge.fit(X_emb_tr, y_train)
preds_gnn_only = np.clip(ridge.predict(X_emb_te), 0.0, 1.0)
 
cent_sc2 = StandardScaler()
X_cent_tr = cent_sc2.fit_transform(features[train_idx])
X_cent_te = cent_sc2.transform(features[test_idx])
xgb_cent  = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    random_state=SEED, n_jobs=-1, verbosity=0,
)
xgb_cent.fit(X_cent_tr, y_train)
preds_cent_only = np.clip(xgb_cent.predict(X_cent_te), 0.0, 1.0)
 
y_test_b = y_all[idx_test]
 
def _get_cent_arr(cent_dict_key, normalizer=1.0):
    return np.array([
        cent_fb[cent_dict_key][nodes[i]] / normalizer
        for i in idx_test
    ], dtype=np.float64)
 
baselines = {
    'Degree'          : _get_cent_arr('deg'),
    'Betweenness'     : _get_cent_arr('bet'),
    'Closeness'       : _get_cent_arr('clo'),
    'Katz'            : _get_cent_arr('katz'),
    'K-Shell'         : _get_cent_arr('ks', max_ks),
    'Harmonic'        : _get_cent_arr('harm', max_harm),
    'GNN Only'        : preds_gnn_only,
    'Centrality Only' : preds_cent_only,
    'Our Model'       : preds_test_eval,
}
 
header = f"\n  {'Method':<20} {'Spearman':>10} {'Kendall':>9} {'R2':>8} {'NDCG@100':>10} {'Jac@100':>9} {'MAP':>8}"
print(header)
print('  ' + '-' * 78)
 
comp_rows = []
for bname, pred in baselines.items():
    pred_c = np.clip(pred.astype(np.float64), 0.0, 1.0)
    s,  _  = spearmanr(y_test_b, pred_c)
    kt_, _ = kendalltau(y_test_b, pred_c)
    r2_b   = r2_score(y_test_b, pred_c)
    nd     = ndcg_at_k(y_test_b, pred_c, 100)
    j      = top_k_jaccard(y_test_b, pred_c, 100, idx_test)
    map_b  = average_precision(y_test_b, pred_c)
    marker = ' ◄ OURS' if bname == 'Our Model' else ''
    print(
        f'  {bname:<20} {s:>10.4f} {kt_:>9.4f} {r2_b:>8.4f}'
        f' {nd:>10.4f} {j:>9.4f} {map_b:>8.4f}{marker}'
    )
    comp_rows.append({
        'Method': bname, 'Spearman': round(s, 4),
        'Kendall': round(kt_, 4), 'R2': round(r2_b, 4),
        'NDCG@100': round(nd, 4), 'Jaccard@100': round(j, 4),
        'MAP': round(map_b, 4),
    })
 
comparison_df = pd.DataFrame(comp_rows).set_index('Method')
comparison_df.to_csv(OUT_DIR / 'baseline_comparison.csv')
log.info('Section 12 complete ✓')
print('Section 12 complete')

05:25:54  INFO      ============================================================
05:25:54  INFO      SECTION 12: Baseline Comparisons
05:25:54  INFO      ============================================================

  Method                 Spearman   Kendall       R2   NDCG@100   Jac@100      MAP
  ------------------------------------------------------------------------------
  Degree                   0.7252    0.5309 -16.8383     0.9237    0.2121   0.9800
  Betweenness              0.5194    0.3823 -17.4436     0.9030    0.1834   0.9465
  Closeness                0.7441    0.5675  -6.1429     0.9602    0.4286   0.9858
  Katz                     0.7757    0.5774 -10.8496     0.9132    0.2048   0.9859
  K-Shell                  0.7452    0.5501  -8.0991     0.9063    0.1976   0.9828
  Harmonic                 0.8012    0.6236  -0.3904     0.9642    0.4706   0.9901
  GNN Only                -0.0266   -0.0168  -0.8593     0.7897    0.0753   0.8871
  Centrality Only         -0.0287   -0.

SECTION 13 — TOP 20 INFLUENTIAL NODES

In [ ]:
log.info('=' * 60)
log.info('SECTION 13: Top 20 Influential Nodes')
log.info('=' * 60)
 
ranking = sorted(
    zip(nodes, preds_all, labels),
    key=lambda x: x[1], reverse=True
)
 
print(
    f"\n  {'Rank':<6} {'Node':>6} {'Pred':>10} {'True':>10} "
    f"{'Degree':>8} {'K-Shell':>8} {'Katz':>8}"
)
print('  ' + '-' * 64)
for rank, (nd, ps, ts) in enumerate(ranking[:20], 1):
    print(
        f'  {rank:<6} {nd:>6} {ps:>10.5f} {ts:>10.5f} '
        f'{G.degree(nd):>8} {ks_c[nd]:>8} {katz_c[nd]:>8.4f}'
    )
 
ranking_df = pd.DataFrame([
    {'rank': r, 'node': nd, 'predicted_score': ps, 'true_epi_score': ts,
     'degree': G.degree(nd), 'kshell': ks_c[nd], 'katz': katz_c[nd],
     'closeness': clo_c[nd], 'betweenness': bet_c[nd],
     'clustering': cl_c[nd], 'harmonic': harm_c[nd]}
    for r, (nd, ps, ts) in enumerate(ranking, 1)
])
ranking_df.to_csv(OUT_DIR / 'ranked_nodes.csv', index=False)
log.info(f'Saved ranked_nodes.csv ({len(ranking_df)} rows)')
log.info('Section 13 complete ✓')
print('Section 13 complete')

05:27:30  INFO      ============================================================
05:27:30  INFO      SECTION 13: Top 20 Influential Nodes
05:27:30  INFO      ============================================================

  Rank     Node       Pred       True   Degree  K-Shell     Katz
  ----------------------------------------------------------------
  1         348    0.86420    0.89169      229       31   0.2229
  2         353    0.86420    0.86751      102       31   0.1619
  3         376    0.86420    0.96159      133       31   0.1776
  4         414    0.86420    0.81593      159       31   0.1837
  5         428    0.86420    0.92276      115       31   0.1718
  6         475    0.86420    0.85045      122       31   0.1722
  7         483    0.86420    0.88128      231       35   0.2448
  8         526    0.86420    0.95473       98       39   0.1790
  9         563    0.86420    0.93797       91       30   0.1532
  10        566    0.86420    0.92013       85       31   0.152

SECTION 14 — VISUALIZATIONS

In [ ]:
log.info('=' * 60)
log.info('SECTION 14A: Visualizations — Training & Evaluation Figures')
log.info('=' * 60)
 
def _save(fig, fname, tight=True):
    path = OUT / fname
    if tight:
        fig.savefig(str(path), dpi=150, bbox_inches='tight')
    else:
        fig.savefig(str(path), dpi=150)
    plt.close(fig)
    log.info(f'  Saved {fname}')
 
 
fig, ax = plt.subplots(figsize=(9, 5))
ep = np.arange(len(train_losses))
ax.plot(ep, train_losses, color=C[0], lw=2.0, label='Train MSE',  zorder=3)
ax.plot(ep, val_losses,   color=C[1], lw=2.0, ls='--', label='Val MSE', zorder=3)
ax.fill_between(ep, train_losses, val_losses, alpha=0.10, color=C[1])
ax.set_title('GNN Training & Validation Loss', fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
_save(fig, 'fig01_training_curve.png')
 
 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Predicted vs Actual Influence Score', fontsize=13, fontweight='bold')
 
for ax, (tag, y_t, y_p) in zip(axes, [
    ('Test Set',  y_test_b,    preds_test_eval),
    ('All Nodes', labels,      preds_all),
]):
    sc = ax.scatter(y_t, y_p, alpha=0.35, s=8,
                    c=y_t, cmap='viridis', edgecolors='none')
    lo = min(float(y_t.min()), float(y_p.min()))
    hi = max(float(y_t.max()), float(y_p.max()))
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.8, label='Ideal', zorder=5)
    sp_loc, _ = spearmanr(y_t, y_p)
    r2_loc    = r2_score(y_t, y_p)
    ax.set_title(
        f'{tag}  (ρ={sp_loc:.4f},  R²={r2_loc:.4f})',
        fontsize=10)
    ax.set_xlabel('Actual Score'); ax.set_ylabel('Predicted Score')
    ax.legend(fontsize=9)
    plt.colorbar(sc, ax=ax, label='Actual Score', shrink=0.85)
 
plt.tight_layout()
_save(fig, 'fig02_predicted_vs_actual.png')
 
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Epidemic Influence Score Distributions', fontsize=13, fontweight='bold')
 
for ax, (arr, title, col) in zip(axes, [
    (labels,     'True Epidemic Labels',  C[2]),
    (preds_all,  'Predicted Scores',      C[0]),
]):
    ax.hist(arr, bins=70, color=col, edgecolor='white', alpha=0.85, density=True)
    ax.axvline(float(np.mean(arr)), color='red', ls='--', lw=2,
               label=f'Mean={np.mean(arr):.4f}')
    ax.axvline(float(np.median(arr)), color='orange', ls=':', lw=2,
               label=f'Median={np.median(arr):.4f}')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Score'); ax.set_ylabel('Density')
    ax.legend(fontsize=9)
 
plt.tight_layout()
_save(fig, 'fig03_label_distribution.png')
 
 
importances = xgb_model.feature_importances_
feat_labels = feature_names + [f'emb_{i}' for i in range(embeddings.shape[1])]
top15_idx   = np.argsort(importances)[-15:]
bar_colors  = [C[1] if 'emb_' in feat_labels[i] else C[0] for i in top15_idx]
 
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    [feat_labels[i] for i in top15_idx],
    importances[top15_idx],
    color=bar_colors, edgecolor='white', height=0.65
)
for bar, val in zip(bars, importances[top15_idx]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)
ax.set_title('Top-15 Feature Importances (XGBoost Gain)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Feature Importance (Gain)')
ax.legend(handles=[
    mpatches.Patch(facecolor=C[0], label='Centrality Feature'),
    mpatches.Patch(facecolor=C[1], label='GNN Embedding'),
], fontsize=9, loc='lower right')
plt.tight_layout()
_save(fig, 'fig04_feature_importance.png')
 
 
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Method Comparison Across Metrics', fontsize=13, fontweight='bold')
 
methods = [r['Method'] for r in comp_rows]
bar_c   = [C[1] if m == 'Our Model' else PALETTE['blue'] for m in methods]
 
for ax, metric, label in zip(axes,
    ['Spearman', 'NDCG@100', 'R2'],
    ['Spearman ρ', 'NDCG@100', 'R²']
):
    vals = [r[metric] for r in comp_rows]
    bars = ax.bar(methods, vals, color=bar_c, edgecolor='white', alpha=0.88)
    ax.set_title(label, fontsize=11)
    ax.set_ylabel(label)
    ax.set_ylim(0, min(1.1, max(vals) * 1.25))
    ax.tick_params(axis='x', rotation=40)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)
    ax.legend(handles=[
        mpatches.Patch(facecolor=C[1], label='Our Model'),
        mpatches.Patch(facecolor=PALETTE['blue'], label='Baselines'),
    ], fontsize=8)
 
plt.tight_layout()
_save(fig, 'fig05_method_comparison.png')
 
 
fig, ax = plt.subplots(figsize=(9, 7))
cent_df = pd.DataFrame(features, columns=feature_names)
corr    = cent_df.corr()
mask    = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, ax=ax, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.6, linecolor='white',
    annot_kws={'fontsize': 9}
)
ax.set_title('Centrality Feature Correlation Matrix',
             fontsize=12, fontweight='bold', pad=10)
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
_save(fig, 'fig06_centrality_correlation.png')
 
 
top30_idx_  = [node_idx[r[0]] for r in ranking[:30]]
top30_cent  = pd.DataFrame(
    features[top30_idx_],
    columns=feature_names,
    index=[f'Node {r[0]}' for r in ranking[:30]]
)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    top30_cent, ax=ax, cmap='Blues', linewidths=0.25,
    linecolor='white', annot=False,
    yticklabels=True, xticklabels=True
)
ax.set_title('Centrality Profile — Top-30 Predicted Influential Nodes',
             fontsize=11, fontweight='bold', pad=10)
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
_save(fig, 'fig07_top30_centrality_heatmap.png')
 
 
fig, ax = plt.subplots(figsize=(12, 10))
top30_nodes = [r[0] for r in ranking[:30]]
sub_g       = G.subgraph(top30_nodes)
pos         = nx.spring_layout(sub_g, seed=SEED, k=0.55)
sc_map      = {r[0]: r[1] for r in ranking[:30]}
max_sc      = max(sc_map.values()) or 1.0
node_sizes  = [(sc_map[nd] / max_sc) * 1600 + 200 for nd in sub_g.nodes()]
node_colors = [sc_map[nd] for nd in sub_g.nodes()]
 
nx.draw_networkx_edges(sub_g, pos, ax=ax, alpha=0.3,
                       edge_color='gray', width=1.0)
sc3 = nx.draw_networkx_nodes(
    sub_g, pos, ax=ax,
    node_size=node_sizes, node_color=node_colors,
    cmap=plt.cm.YlOrRd, alpha=0.92)
nx.draw_networkx_labels(sub_g, pos, ax=ax, font_size=7,
                        font_weight='bold', font_color='black')
plt.colorbar(sc3, ax=ax, label='Predicted Influence Score', shrink=0.75)
ax.set_title(
    f'Top-30 Influential Nodes Subgraph\n'
    f'(node size ∝ predicted influence)',
    fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
_save(fig, 'fig08_top30_subgraph.png')
 
 
TOP_CMP = 50
true_ranks_s = pd.Series(y_test_b).rank(ascending=False)
sorted_by_true = np.argsort(y_test_b)[::-1][:TOP_CMP]
 
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(range(TOP_CMP),
        [true_ranks_s.iloc[i] for i in sorted_by_true],
        'k--', lw=2.5, label='True Rank', zorder=6)
 
cmp_preds = {
    'Our Model (Hybrid)' : preds_test_eval,
    'GNN Only'           : preds_gnn_only,
    'Centrality Only'    : preds_cent_only,
    'Katz'               : np.array([katz_c[nodes[i]] for i in idx_test]),
    'Degree'             : np.array([deg_c[nodes[i]]  for i in idx_test]),
}
for ci, (nm, pred) in enumerate(cmp_preds.items()):
    pr_s = pd.Series(np.clip(pred, 0, 1)).rank(ascending=False)
    lw   = 2.5 if 'Our Model' in nm else 1.5
    ax.plot(range(TOP_CMP),
            [pr_s.iloc[i] for i in sorted_by_true],
            color=C[ci], lw=lw, alpha=0.85, label=nm)
 
ax.set_xlabel(f'True Rank (Top {TOP_CMP} nodes)')
ax.set_ylabel('Method-Predicted Rank')
ax.set_title(
    f'Rank-Order Comparison — Top-{TOP_CMP} Nodes',
    fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
_save(fig, 'fig09_ranking_comparison.png')
 
 
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Top-20 Most Influential Nodes', fontsize=13, fontweight='bold')
 
top20 = ranking[:20]
t20_names  = [str(r[0]) for r in top20]
t20_pred   = [r[1] for r in top20]
t20_true   = [r[2] for r in top20]
 
cmap20 = plt.cm.YlOrRd(np.linspace(0.3, 0.95, 20))
 
ax = axes[0]
ax.barh(t20_names[::-1], t20_pred[::-1],
        color=cmap20, edgecolor='white', height=0.7)
ax.set_title('Predicted Influence Score')
ax.set_xlabel('Predicted Score')
 
ax = axes[1]
ax.barh(t20_names[::-1], t20_true[::-1],
        color=cmap20, edgecolor='white', height=0.7)
ax.set_title('True Epidemic Score')
ax.set_xlabel('True Score')
 
plt.tight_layout()
_save(fig, 'fig10_top20_nodes.png')
 
 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('5-Fold Cross-Validation Results', fontsize=13, fontweight='bold')
 
metrics_cv   = ['spearman', 'kendall', 'r2', 'mse', 'ndcg50', 'jaccard50']
cv_means = [np.mean(cv_results[m]) for m in metrics_cv]
cv_stds  = [np.std(cv_results[m])  for m in metrics_cv]
 
ax = axes[0]
bars = ax.bar(metrics_cv, cv_means, yerr=cv_stds,
              color=C[:len(metrics_cv)], edgecolor='white',
              capsize=6, alpha=0.85)
ax.set_title('Mean ± Std across 5 Folds')
ax.set_ylabel('Metric Value'); ax.tick_params(axis='x', rotation=30)
for bar, mean, std in zip(bars, cv_means, cv_stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + std + 0.005,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=8)
 
ax = axes[1]
for ci, metric in enumerate(['spearman', 'ndcg50', 'r2']):
    ax.plot(range(1, 6), cv_results[metric],
            marker='o', lw=2, color=C[ci], label=metric)
ax.set_title('Fold-by-Fold Performance')
ax.set_xlabel('Fold'); ax.set_ylabel('Metric Value')
ax.set_xticks(range(1, 6)); ax.legend(fontsize=9)
 
plt.tight_layout()
_save(fig, 'fig11_cv_results.png')
 
 
log.info('Section 14A complete ✓')
print('Section 14A complete')

05:31:29  INFO      ============================================================
05:31:29  INFO      SECTION 14A: Visualizations — Training & Evaluation Figures
05:31:29  INFO      ============================================================
05:31:29  INFO        Saved fig01_training_curve.png
05:31:30  INFO        Saved fig02_predicted_vs_actual.png
05:31:31  INFO        Saved fig03_label_distribution.png
05:31:31  INFO        Saved fig04_feature_importance.png
05:31:32  INFO        Saved fig05_method_comparison.png
05:31:33  INFO        Saved fig06_centrality_correlation.png
05:31:34  INFO        Saved fig07_top30_centrality_heatmap.png
05:31:34  INFO        Saved fig08_top30_subgraph.png
05:31:34  INFO        Saved fig09_ranking_comparison.png
05:31:34  INFO      Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
05:31:34  INFO      Using categor

SECTION 15 — MULTI-DATASET GENERALIZATION (Zero-Shot)

In [ ]:
log.info('=' * 60)
log.info('SECTION 15: Multi-Dataset Generalization (Zero-Shot)')
log.info('=' * 60)
 
 
def infer_on_graph(
        G_new: nx.Graph,
        ds_name: str,
        gnn_model: nn.Module,
        xgb_mdl,
        feat_scaler_fitted,
        fb_feat_dim: int,
        epi_cfg: dict,
        seed: int = 42,
) -> dict:
    nodes_new = sorted(G_new.nodes())
    N_new     = G_new.number_of_nodes()
    safe      = ds_name.lower().replace('-', '').replace(' ', '_')
    log.info(
        f'[{ds_name}] Nodes={N_new:,}  Edges={G_new.number_of_edges():,}'
    )
 
    feats_raw_new, _, _ = compute_centralities(G_new, name=safe, seed=seed)
    local_sc   = MinMaxScaler()
    feats_new  = local_sc.fit_transform(feats_raw_new).astype(np.float32)
    feats_new  = np.clip(feats_new, 0.0, 1.0)
 
    n_f = feats_new.shape[1]
    if n_f < fb_feat_dim:
        pad       = np.zeros((N_new, fb_feat_dim - n_f), dtype=np.float32)
        feats_new = np.concatenate([feats_new, pad], axis=1)
    elif n_f > fb_feat_dim:
        feats_new = feats_new[:, :fb_feat_dim]
 
    data_new   = from_networkx(G_new).to(DEVICE)
    data_new.x = torch.tensor(feats_new, dtype=torch.float, device=DEVICE)
    gnn_model.eval()
    with torch.no_grad():
        emb_new_t, _ = gnn_model(data_new)
        emb_new      = emb_new_t.cpu().numpy()
 
    X_new    = np.concatenate([feats_new, emb_new], axis=1)
    X_new_sc = feat_scaler_fitted.transform(X_new)
    preds_new = np.clip(xgb_mdl.predict(X_new_sc), 0.0, 1.0)
 
    labels_new = compute_epidemic_labels(
        G_in       = G_new,
        nodes_list = nodes_new,
        betas      = epi_cfg['EPI_BETAS'],
        gamma      = epi_cfg['EPI_GAMMA'],
        sigma      = epi_cfg['EPI_SIGMA'],
        steps      = epi_cfg['EPI_STEPS_OTHER'],
        runs       = epi_cfg['EPI_RUNS_OTHER'],
        cache_name = safe,
        n_jobs     = epi_cfg['EPI_N_JOBS'],
    )
 
    gidx   = np.arange(N_new)
    sp_, _ = spearmanr(labels_new, preds_new)
    kt_, _ = kendalltau(labels_new, preds_new)
    r2_    = r2_score(labels_new, preds_new)
    nd50   = ndcg_at_k(labels_new, preds_new, 50)
    jac50  = top_k_jaccard(labels_new, preds_new, 50, gidx)
    map_   = average_precision(labels_new, preds_new)
 
    print(f'\n[{ds_name}]')
    print(f'  Spearman   : {sp_:.4f}')
    print(f'  Kendall    : {kt_:.4f}')
    print(f'  R²         : {r2_:.4f}')
    print(f'  NDCG@50    : {nd50:.4f}')
    print(f'  Jaccard@50 : {jac50:.4f}')
    print(f'  MAP        : {map_:.4f}')
 
    row = {
        'dataset': ds_name, 'nodes': N_new,
        'edges': G_new.number_of_edges(),
        'spearman': sp_, 'kendall': kt_, 'r2': r2_,
        'ndcg_50': nd50, 'jaccard_50': jac50, 'map': map_,
    }
    pd.DataFrame([row]).to_csv(
        OUT_DIR / f'{safe}_results.csv', index=False)
    return row
 
 
gen_results = [{
    'dataset': 'Facebook (train)', 'nodes': N, 'edges': E,
    'spearman': spear, 'kendall': ktau, 'r2': r2,
    'ndcg_50': main_metrics.get('ndcg@50', 0.0),
    'jaccard_50': main_metrics.get('jaccard@50', 0.0),
    'map': main_metrics.get('map', 0.0),
}]
 
res_em = infer_on_graph(
    G_em, 'Email-EU', model_gnn, xgb_model, feat_scaler,
    features.shape[1], CFG
)
gen_results.append(res_em)
 
res_wv = infer_on_graph(
    G_wv, 'WikiVote', model_gnn, xgb_model, feat_scaler,
    features.shape[1], CFG
)
gen_results.append(res_wv)
 
print('\n' + '=' * 70)
print('MULTI-DATASET GENERALIZATION RESULTS')
print('=' * 70)
for r in gen_results:
    print(
        f"{r['dataset']:<25} | ρ={r['spearman']:.4f} | "
        f"τ={r['kendall']:.4f} | R²={r['r2']:.4f} | "
        f"NDCG@50={r['ndcg_50']:.4f} | MAP={r['map']:.4f}"
    )
 
gen_df = pd.DataFrame(gen_results)
gen_df.to_csv(OUT_DIR / 'multi_dataset_results.csv', index=False)
log.info('Section 15 complete ✓')
print('Section 15 complete')

05:32:21  INFO      ============================================================
05:32:21  INFO      SECTION 15: Multi-Dataset Generalization (Zero-Shot)
05:32:21  INFO      ============================================================
05:32:21  INFO      [Email-EU] Nodes=986  Edges=16,064
05:32:21  INFO      [emaileu] [1/8] Degree centrality...
05:32:21  INFO      [emaileu] [2/8] Betweenness (k=100)...
05:32:21  INFO      [emaileu] [3/8] Closeness centrality...
05:32:22  INFO      [emaileu] [4/8] Katz centrality (numpy)...
05:32:23  INFO      [emaileu] [5/8] K-shell (core number)...
05:32:23  INFO      [emaileu] [6/8] Clustering coefficient...
05:32:23  INFO      [emaileu] [7/8] VoteRank...
05:32:27  INFO      [emaileu] [8/8] Harmonic centrality...
05:32:28  INFO      [emaileu] Centralities done in 7.4s  → cached
05:32:29  INFO      [emaileu] Simulating 986 nodes  models=SIR+SIS+SEIR  β=[0.1, 0.2, 0.3]  steps=8  runs=8  jobs=-1


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:   11.0s
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:   24.4s
[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:   42.4s
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 640 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 874 tasks      | elapsed:  1.9min


05:34:32  INFO      [emaileu] Labels done in 123.2s  mean=0.7429  std=0.1861  range=[0.0000, 1.0000]

[Email-EU]
  Spearman   : 0.7795
  Kendall    : 0.6277
  R²         : 0.8288
  NDCG@50    : 0.9031
  Jaccard@50 : 0.0638
  MAP        : 0.9992
05:34:32  INFO      [WikiVote] Nodes=7,066  Edges=100,736
05:34:32  INFO      [wikivote] [1/8] Degree centrality...
05:34:32  INFO      [wikivote] [2/8] Betweenness (k=100)...


[Parallel(n_jobs=-1)]: Done 986 out of 986 | elapsed:  2.1min finished


05:34:37  INFO      [wikivote] [3/8] Closeness centrality...
05:35:41  INFO      [wikivote] [4/8] Katz centrality (numpy)...
05:35:47  INFO      [wikivote] [5/8] K-shell (core number)...
05:35:48  INFO      [wikivote] [6/8] Clustering coefficient...
05:35:51  INFO      [wikivote] [7/8] VoteRank...
05:37:37  INFO      [wikivote] [8/8] Harmonic centrality...
05:38:46  INFO      [wikivote] Centralities done in 254.3s  → cached
05:38:49  INFO      [wikivote] Simulating 7,066 nodes  models=SIR+SIS+SEIR  β=[0.1, 0.2, 0.3]  steps=8  runs=8  jobs=-1


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   11.5s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:  3.6min
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed:  5.6min
[Parallel(n_jobs=-1)]: Done 640 tasks      | elapsed:  7.7min
[Parallel(n_jobs=-1)]: Done 874 tasks      | elapsed: 10.7min
[Parallel(n_jobs=-1)]: Done 1144 tasks      | elapsed: 14.0min
[Parallel(n_jobs=-1)]: Done 1450 tasks      | elapsed: 17.5min
[Parallel(n_jobs=-1)]: Done 1792 tasks      | elapsed: 21.5min
[Parallel(n_jobs=-1)]: Done 2170 tasks      | elapsed: 26.0min
[Parallel(n_jobs=-1)]: Done 2584 tasks      | elapsed: 31.1min
[Parallel(n_jobs=-1)]: Done 3034 tasks      | elapsed: 36.2min
[Parallel(n_jobs=-1)]: Done 3520 tasks      | elapsed: 41.8min
[Parallel(n_jobs=-1)]: Done 4042 tasks      | ela

06:58:26  INFO      [wikivote] Labels done in 4776.6s  mean=0.6217  std=0.2266  range=[0.0000, 1.0000]

[WikiVote]
  Spearman   : 0.9288
  Kendall    : 0.7654
  R²         : 0.8984
  NDCG@50    : 0.8866
  Jaccard@50 : 0.0101
  MAP        : 0.9959

MULTI-DATASET GENERALIZATION RESULTS
Facebook (train)          | ρ=0.9169 | τ=0.7583 | R²=0.9228 | NDCG@50=0.9621 | MAP=0.9995
Email-EU                  | ρ=0.7795 | τ=0.6277 | R²=0.8288 | NDCG@50=0.9031 | MAP=0.9992
WikiVote                  | ρ=0.9288 | τ=0.7654 | R²=0.8984 | NDCG@50=0.8866 | MAP=0.9959
06:58:26  INFO      Section 15 complete ✓
Section 15 complete


[Parallel(n_jobs=-1)]: Done 7066 out of 7066 | elapsed: 79.6min finished


SECTION 16 — STATISTICAL SIGNIFICANCE (Bonferroni-Corrected)

In [ ]:
log.info('=' * 60)
log.info('SECTION 16: Bonferroni-Corrected Wilcoxon Signed-Rank Tests')
log.info('=' * 60)
 
 
def abs_rank_error(y_true, y_pred):
    return np.abs(
        np.argsort(np.argsort(y_true)[::-1]) -
        np.argsort(np.argsort(y_pred)[::-1])
    ).astype(float)
 
 
our_errors  = abs_rank_error(y_test_b, preds_test_eval)
sig_methods = {
    'Degree'          : np.clip(_get_cent_arr('deg'),         0, 1),
    'Betweenness'     : np.clip(_get_cent_arr('bet'),         0, 1),
    'Closeness'       : np.clip(_get_cent_arr('clo'),         0, 1),
    'Katz'            : np.clip(_get_cent_arr('katz'),        0, 1),
    'K-Shell'         : np.clip(_get_cent_arr('ks', max_ks),  0, 1),
    'Harmonic'        : np.clip(_get_cent_arr('harm', max_harm), 0, 1),
    'GNN Only'        : preds_gnn_only,
    'Centrality Only' : preds_cent_only,
}
 
n_tests = len(sig_methods)
alpha   = CFG['ALPHA']
bonf    = alpha / n_tests
 
print(f'\n  α={alpha}  n_tests={n_tests}  Bonferroni threshold={bonf:.4f}')
print(
    f"\n  {'Baseline':<20} {'W-stat':>10} {'p-value':>12} "
    f"{'p<α':>7} {'p<Bonf':>9} {'Significant':>12}"
)
print('  ' + '-' * 74)
 
sig_rows = []
for bsname, bl_pred in sig_methods.items():
    bl_err = abs_rank_error(y_test_b, np.array(bl_pred))
    try:
        stat, pval = wilcoxon(our_errors, bl_err, alternative='less')
    except ValueError:   # identical arrays
        stat, pval = 0.0, 1.0
    raw_s  = '✓' if pval < alpha else '✗'
    bonf_s = '✓' if pval < bonf  else '✗'
    sig_s  = 'YES' if pval < bonf else ('MARGINAL' if pval < alpha else 'NO')
    print(
        f'  {bsname:<20} {stat:>10.1f} {pval:>12.4e} '
        f'{raw_s:>7} {bonf_s:>9} {sig_s:>12}'
    )
    sig_rows.append({
        'baseline': bsname, 'W': stat, 'p': pval,
        'sig_alpha': pval < alpha, 'sig_bonferroni': pval < bonf,
    })
 
sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(OUT_DIR / 'significance_tests.csv', index=False)
log.info('Section 16 complete ✓')
print('Section 16 complete')

07:02:44  INFO      ============================================================
07:02:44  INFO      SECTION 16: Bonferroni-Corrected Wilcoxon Signed-Rank Tests
07:02:44  INFO      ============================================================

  α=0.05  n_tests=8  Bonferroni threshold=0.0063

  Baseline                 W-stat      p-value     p<α    p<Bonf  Significant
  --------------------------------------------------------------------------
  Degree                  64318.0   7.3637e-50       ✓         ✓          YES
  Betweenness             42844.0   1.2293e-73       ✓         ✓          YES
  Closeness               72445.5   5.9280e-41       ✓         ✓          YES
  Katz                    75127.0   1.2288e-39       ✓         ✓          YES
  K-Shell                 66849.5   9.1460e-47       ✓         ✓          YES
  Harmonic                83115.5   2.1889e-32       ✓         ✓          YES
  GNN Only                22877.0   1.2189e-99       ✓         ✓          YES
  Cent

SECTION 17 — ABLATION STUDY

In [ ]:
log.info('=' * 60)
log.info('SECTION 17: Ablation Study')
log.info('=' * 60)
 
n_cent = features.shape[1]
n_emb  = embeddings.shape[1]
 
ablation_configs = {
    f'Centrality Only ({n_cent} feat)' : (X_fb_all[:, :n_cent],         False),
    f'GNN Emb Only ({n_emb} feat)'     : (X_fb_all[:, n_cent:],         False),
    f'No Residual (GNN ablation)'      : (X_fb_all,                     False),
    f'Hybrid ({n_cent+n_emb} feat)'    : (X_fb_all,                     True),
}
 
print(f"\n  {'Config':<38} {'Spearman':>10} {'NDCG@100':>10} {'R2':>8} {'MAP':>8}")
print('  ' + '-' * 74)
 
abl_rows = []
for cfg_name, (X_cfg, is_ours) in ablation_configs.items():
    abl_sc = StandardScaler()
    Xtr_a  = abl_sc.fit_transform(X_cfg[train_idx])
    Xte_a  = abl_sc.transform(X_cfg[test_idx])
    xgb_a  = XGBRegressor(
        n_estimators=350, max_depth=5, learning_rate=0.05,
        random_state=SEED, n_jobs=-1, verbosity=0)
    xgb_a.fit(Xtr_a, y_train)
    p_a    = np.clip(xgb_a.predict(Xte_a), 0.0, 1.0)
    sp_a, _ = spearmanr(y_test_b, p_a)
    nd_a    = ndcg_at_k(y_test_b, p_a, 100)
    r2_a    = r2_score(y_test_b, p_a)
    map_a   = average_precision(y_test_b, p_a)
    marker  = ' ◄ OURS' if is_ours else ''
    print(f'  {cfg_name:<38} {sp_a:>10.4f} {nd_a:>10.4f} {r2_a:>8.4f} {map_a:>8.4f}{marker}')
    abl_rows.append({
        'Config': cfg_name, 'Spearman': round(sp_a, 4),
        'NDCG@100': round(nd_a, 4), 'R2': round(r2_a, 4),
        'MAP': round(map_a, 4), 'Our Model': is_ours,
    })
 
pd.DataFrame(abl_rows).to_csv(OUT_DIR / 'ablation_study.csv', index=False)
log.info('Section 17 complete ✓')
print('Section 17 complete')

07:02:50  INFO      ============================================================
07:02:50  INFO      SECTION 17: Ablation Study
07:02:50  INFO      ============================================================

  Config                                   Spearman   NDCG@100       R2      MAP
  --------------------------------------------------------------------------
  Centrality Only (8 feat)                  -0.0311     0.7945  -0.9857   0.8894
  GNN Emb Only (32 feat)                    -0.0158     0.7995  -0.9170   0.8951
  No Residual (GNN ablation)                -0.0212     0.8009  -0.9682   0.8920
  Hybrid (40 feat)                          -0.0212     0.8009  -0.9682   0.8920 ◄ OURS
07:02:53  INFO      Section 17 complete ✓
Section 17 complete


SECTION 18 — HYPERPARAMETER SENSITIVITY

In [ ]:
log.info('=' * 60)
log.info('SECTION 18: Hyperparameter Sensitivity')
log.info('=' * 60)
 
HIDDEN_VALS  = [32, 64, 128]
DROPOUT_VALS = [0.1, 0.2, 0.3, 0.4]
HEAD_VALS    = [2, 4]
sens_rows    = []
 
for hidden in HIDDEN_VALS:
    for heads in HEAD_VALS:
        if hidden % heads != 0:
            continue
        for dropout in DROPOUT_VALS:
            try:
                m_s = HybridGNN(
                    in_dim=features.shape[1], hidden=hidden,
                    out_dim=32, heads=heads, dropout=dropout,
                ).to(DEVICE)
                o_s = optim.Adam(m_s.parameters(), lr=5e-3, weight_decay=1e-4)
                m_s.train()
                for _ in range(80):
                    o_s.zero_grad()
                    _, sc_s = m_s(data)
                    F.mse_loss(
                        sc_s[data.train_mask], data.y[data.train_mask]
                    ).backward()
                    torch.nn.utils.clip_grad_norm_(m_s.parameters(), 1.0)
                    o_s.step()
                m_s.eval()
                with torch.no_grad():
                    e_s, _ = m_s(data)
                X_s   = np.concatenate([features, e_s.cpu().numpy()], axis=1)
                sc2   = StandardScaler()
                xb_s  = XGBRegressor(
                    n_estimators=200, max_depth=5, learning_rate=0.05,
                    random_state=SEED, n_jobs=-1, verbosity=0)
                xb_s.fit(sc2.fit_transform(X_s[train_idx]), y_train)
                p_s   = np.clip(xb_s.predict(sc2.transform(X_s[test_idx])), 0.0, 1.0)
                sp_s, _ = spearmanr(y_test_b, p_s)
                sens_rows.append({
                    'hidden': hidden, 'dropout': dropout,
                    'heads': heads, 'spearman': round(sp_s, 4),
                })
                log.info(
                    f'  Sensitivity: h={hidden} d={dropout} H={heads} '
                    f'ρ={sp_s:.4f}'
                )
                del m_s, o_s
            except Exception as ex:
                log.warning(f'  SKIP h={hidden} d={dropout} H={heads}: {ex}')
 
sens_df = pd.DataFrame(sens_rows) if sens_rows else pd.DataFrame()
if not sens_df.empty:
    best_row = sens_df.loc[sens_df['spearman'].idxmax()]
    print(
        f'\n  Best config: hidden={int(best_row["hidden"])}  '
        f'dropout={best_row["dropout"]}  '
        f'heads={int(best_row["heads"])}  '
        f'ρ={best_row["spearman"]:.4f}'
    )
    sens_df.to_csv(OUT_DIR / 'sensitivity_results.csv', index=False)
 
log.info('Section 18 complete ✓')
print('Section 18 complete')

07:03:51  INFO      ============================================================
07:03:51  INFO      SECTION 18: Hyperparameter Sensitivity
07:03:51  INFO      ============================================================
07:03:52  INFO        Sensitivity: h=32 d=0.1 H=2 ρ=-0.0239
07:03:54  INFO        Sensitivity: h=32 d=0.2 H=2 ρ=-0.0258
07:03:55  INFO        Sensitivity: h=32 d=0.3 H=2 ρ=-0.0271
07:03:56  INFO        Sensitivity: h=32 d=0.4 H=2 ρ=-0.0253
07:03:57  INFO        Sensitivity: h=32 d=0.1 H=4 ρ=-0.0269
07:03:59  INFO        Sensitivity: h=32 d=0.2 H=4 ρ=-0.0272
07:04:00  INFO        Sensitivity: h=32 d=0.3 H=4 ρ=-0.0231
07:04:01  INFO        Sensitivity: h=32 d=0.4 H=4 ρ=-0.0251
07:04:02  INFO        Sensitivity: h=64 d=0.1 H=2 ρ=-0.0243
07:04:04  INFO        Sensitivity: h=64 d=0.2 H=2 ρ=-0.0262
07:04:05  INFO        Sensitivity: h=64 d=0.3 H=2 ρ=-0.0241
07:04:06  INFO        Sensitivity: h=64 d=0.4 H=2 ρ=-0.0212
07:04:08  INFO        Sensitivity: h=64 d=0.1 H=4 ρ=-0.0216

SECTION 19 — RUNTIME ANALYSIS

In [ ]:
log.info('=' * 60)
log.info('SECTION 19: Runtime Analysis')
log.info('=' * 60)
 
runtime_ops = {
    'Degree centrality'         : lambda: nx.degree_centrality(G),
    'Betweenness (k=100)'       : lambda: nx.betweenness_centrality(G, k=100),
    'Closeness centrality'      : lambda: nx.closeness_centrality(G),
    'K-Shell (core)'            : lambda: nx.core_number(G),
    'Katz centrality (numpy)'   : lambda: nx.katz_centrality_numpy(G, alpha=0.005),
    'Harmonic centrality'       : lambda: nx.harmonic_centrality(G),
    'VoteRank'                  : lambda: nx.voterank(G),
    'GNN inference'             : None,   
    'XGBoost inference'         : None,
    'Epidemic labels (cached)'  : None,
}
 
runtime_rows = []
print(f'\n  {"Operation":<30} {"Time (s)":>10}')
print('  ' + '-' * 43)
 
for mname, mfn in runtime_ops.items():
    if mfn is None:
        continue
    t0 = time.time()
    try:
        mfn()
    except Exception:
        pass
    elapsed = time.time() - t0
    print(f'  {mname:<30} {elapsed:>10.3f}s')
    runtime_rows.append({'operation': mname, 'time_sec': round(elapsed, 3), 'stage': 'feature'})
 
t0 = time.time()
with torch.no_grad():
    _, _ = model_gnn(data)
t_gnn_inf = time.time() - t0
print(f'  {"GNN inference":<30} {t_gnn_inf:>10.3f}s')
runtime_rows.append({'operation': 'GNN inference', 'time_sec': round(t_gnn_inf, 3), 'stage': 'inference'})
 
t0 = time.time()
_ = xgb_model.predict(X_fb_all)
t_xgb_inf = time.time() - t0
print(f'  {"XGBoost inference":<30} {t_xgb_inf:>10.3f}s')
runtime_rows.append({'operation': 'XGBoost inference', 'time_sec': round(t_xgb_inf, 3), 'stage': 'inference'})
 
print(f'  {"GNN training":<30} {t_gnn_total:>10.1f}s')
print(f'  {"XGBoost training":<30} {t_xgb_total:>10.1f}s')
print(f'  {"Epidemic simulation":<30} {t_epi_total:>10.1f}s')
runtime_rows.extend([
    {'operation': 'GNN training',        'time_sec': round(t_gnn_total, 2), 'stage': 'training'},
    {'operation': 'XGBoost training',    'time_sec': round(t_xgb_total, 2), 'stage': 'training'},
    {'operation': 'Epidemic simulation', 'time_sec': round(t_epi_total, 2), 'stage': 'label_gen'},
])
 
runtime_df = pd.DataFrame(runtime_rows)
runtime_df.to_csv(OUT_DIR / 'runtime_analysis.csv', index=False)
log.info('Section 19 complete ✓')
print('Section 19 complete')

07:04:28  INFO      ============================================================
07:04:28  INFO      SECTION 19: Runtime Analysis
07:04:28  INFO      ============================================================

  Operation                        Time (s)
  -------------------------------------------
  Degree centrality                   0.002s
  Betweenness (k=100)                 2.451s
  Closeness centrality               22.613s
  K-Shell (core)                      0.082s
  Katz centrality (numpy)             1.645s
  Harmonic centrality                24.994s
  VoteRank                          101.416s
  GNN inference                       0.004s
  XGBoost inference                   0.004s
  GNN training                          1.6s
  XGBoost training                      0.3s
  Epidemic simulation                5748.3s
07:07:02  INFO      Section 19 complete ✓
Section 19 complete


SECTION 20 — SAVE MODEL WEIGHTS & FINAL SUMMARY

In [ ]:
log.info('=' * 60)
log.info('SECTION 20: Save Model Weights & Final Summary')
log.info('=' * 60)
 
gnn_save_path = OUT_DIR / 'hybrid_gnn_weights.pt'
torch.save({
    'model_state_dict': model_gnn.state_dict(),
    'model_config': {
        'in_dim' : features.shape[1],
        'hidden' : CFG['GNN_HIDDEN'],
        'out_dim': CFG['GNN_OUT'],
        'heads'  : CFG['GNN_HEADS'],
        'dropout': CFG['GNN_DROPOUT'],
    },
    'train_idx'    : train_idx,
    'val_idx'      : val_idx,
    'test_idx'     : test_idx,
    'feature_names': feature_names,
    'cfg'          : CFG,
    'metrics': {
        'spearman_test': float(spear),
        'kendall_test':  float(ktau),
        'r2_test':       float(r2),
        'mse_test':      float(mse),
    },
    'seed': SEED,
}, str(gnn_save_path))
log.info(f'GNN saved → {gnn_save_path}')
 
xgb_save_path = OUT_DIR / 'xgboost_model.pkl'
with open(xgb_save_path, 'wb') as fh:
    pickle.dump(xgb_model, fh)
log.info(f'XGBoost saved → {xgb_save_path}')
 
scalers_path = OUT_DIR / 'scalers.pkl'
with open(scalers_path, 'wb') as fh:
    pickle.dump({
        'cent_scaler' : cent_scaler,
        'feat_scaler' : feat_scaler,
    }, fh)
log.info(f'Scalers saved → {scalers_path}')
 
ckpt = torch.load(str(gnn_save_path), map_location=DEVICE, weights_only=False)
m_rl = HybridGNN(**ckpt['model_config']).to(DEVICE)
m_rl.load_state_dict(ckpt['model_state_dict'])
m_rl.eval()
with torch.no_grad():
    _, sc_rl = m_rl(data)
assert sc_rl.min() >= 0.0 and sc_rl.max() <= 1.0, 'Reload score out of [0,1]!'
log.info(
    f'Reload verified: score range=[{sc_rl.min():.4f}, {sc_rl.max():.4f}] ✓'
)
print(f'  Reload verified: range [{sc_rl.min():.4f}, {sc_rl.max():.4f}]')
 
print(f'\n  Output files in {OUT_DIR}/')
for fn in sorted(OUT_DIR.iterdir()):
    sz = fn.stat().st_size / 1024
    print(f'    {fn.name:<48} {sz:>7.1f} KB')
 
log.info('Section 20 complete ✓')
print('Section 20 complete')

07:07:23  INFO      ============================================================
07:07:23  INFO      SECTION 20: Save Model Weights & Final Summary
07:07:23  INFO      ============================================================
07:07:23  INFO      GNN saved → /kaggle/working/results/hybrid_gnn_weights.pt
07:07:23  INFO      XGBoost saved → /kaggle/working/results/xgboost_model.pkl
07:07:23  INFO      Scalers saved → /kaggle/working/results/scalers.pkl
07:07:23  INFO      Reload verified: score range=[0.0802, 0.9142] ✓
  Reload verified: range [0.0802, 0.9142]

  Output files in /kaggle/working/results/
    ablation_study.csv                                   0.3 KB
    baseline_comparison.csv                              0.5 KB
    centrality_feature_stats.csv                         1.0 KB
    cv_results.csv                                       0.6 KB
    dataset_summary.csv                                  0.1 KB
    emaileu_results.csv                                  0.2 KB
    f